# NumpyDataset: атомарный disk cache

## Goal

Показать третий этап реализации датасета: сохранение загруженных EEG/EOG-массивов в `.npy` и повторное чтение без открытия FIF через MNE. Ноутбук проверяет:

- структуру cache entry;
- содержимое `manifest.json`;
- реальный cache hit;
- разделение кэша по dtype;
- автоматический путь `artifacts/cache/...`.

## Setup

Для демонстрации используется отдельный generated-каталог `artifacts/cache/notebook-demo`. Он игнорируется Git и не изменяет исходные FIF.

In [1]:
import json
import sys
import time
from pathlib import Path
from unittest.mock import patch

import numpy as np
import pandas as pd
from IPython.display import display


def find_project_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").is_file():
            return candidate
    raise RuntimeError("Project root with pyproject.toml was not found")


PROJECT_ROOT = find_project_root(Path.cwd().resolve())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from utils.datasets import NumpyDataset

DATASET_DIR = PROJECT_ROOT / "data" / "Data_Train"
DEMO_CACHE_ROOT = PROJECT_ROOT / "artifacts" / "cache" / "notebook-demo"

print(f"Dataset source: {DATASET_DIR}")
print(f"Demo cache: {DEMO_CACHE_ROOT}")

Dataset source: /home/slauva/Projects/master-thesis-2024-2026/code/data/Data_Train
Demo cache: /home/slauva/Projects/master-thesis-2024-2026/code/artifacts/cache/notebook-demo


### Key Assumptions

- Источником истины остаются FIF-файлы.
- Cache hit разрешен только при совпадении `size`, `mtime_ns`, dtype и версии cache schema.
- `manifest.json` записывается последним и служит маркером завершенной записи.

## Steps

### 1. Создать датасет с disk cache

Перед первым чтением cache entry еще не существует.

In [2]:
dataset = NumpyDataset(
    DATASET_DIR,
    dtype=np.float32,
    cache_policy="disk",
    cache_dir=DEMO_CACHE_ROOT / "float32",
)
sample_key = (1, 1, 1)
entry_dir = dataset.get_cache_entry_path(sample_key)

if entry_dir.exists():
    for cache_file in entry_dir.iterdir():
        cache_file.unlink()
    entry_dir.rmdir()

pd.Series(
    {
        "cache_policy": dataset.cache_policy,
        "cache_dir": str(dataset.cache_dir.relative_to(PROJECT_ROOT)),
        "entry_exists_before_load": entry_dir.exists(),
    },
    name="configuration",
)

cache_policy                                                 disk
cache_dir                   artifacts/cache/notebook-demo/float32
entry_exists_before_load                                    False
Name: configuration, dtype: object

### 2. Первый доступ: FIF → NumPy → cache

При cache miss загрузчик читает EEG/EOG через MNE и атомарно записывает два массива и manifest.

In [3]:
started_at = time.perf_counter()
first_loaded = dataset[sample_key]
first_load_seconds = time.perf_counter() - started_at

cache_files = sorted(path.name for path in entry_dir.iterdir())
cache_file_summary = pd.DataFrame(
    [
        {"file": path.name, "size_mib": path.stat().st_size / 2**20}
        for path in sorted(entry_dir.iterdir())
    ]
)
display(cache_file_summary)
print(f"First load: {first_load_seconds:.4f} s")

,file,size_mib
0,eeg.npy,3.845577
1,eog.npy,0.305317
2,manifest.json,0.001511


First load: 0.0284 s


### 3. Прочитать manifest

Manifest связывает массивы с конкретными исходными файлами и параметрами загрузки.

In [4]:
with (entry_dir / "manifest.json").open(encoding="utf-8") as file:
    manifest = json.load(file)

manifest_view = pd.Series(
    {
        "schema_version": manifest["schema_version"],
        "key": manifest["key"],
        "dtype": manifest["dtype"],
        "eeg_shape": manifest["arrays"]["eeg"]["shape"],
        "eog_shape": manifest["arrays"]["eog"]["shape"],
        "sfreq": manifest["sfreq"],
        "eeg_source_size": manifest["sources"]["eeg"]["size"],
        "eog_source_size": manifest["sources"]["eog"]["size"],
    },
    name="manifest",
)
manifest_view

schema_version                                                     1
key                {'subject_id': 1, 'trial_number': 1, 'block_in...
dtype                                                        float32
eeg_shape                                                [63, 16001]
eog_shape                                                 [5, 16001]
sfreq                                                         1000.0
eeg_source_size                                              4043200
eog_source_size                                               321340
Name: manifest, dtype: object

### 4. Второй доступ: доказать cache hit

На время второго чтения запрещаем `mne.io.read_raw_fif`. Успешная загрузка означает, что данные пришли из `.npy`.

In [5]:
with patch("mne.io.read_raw_fif", side_effect=AssertionError("Unexpected FIF read")):
    started_at = time.perf_counter()
    cached_loaded = dataset[sample_key]
    cached_load_seconds = time.perf_counter() - started_at

np.testing.assert_array_equal(cached_loaded.eeg, first_loaded.eeg)
np.testing.assert_array_equal(cached_loaded.eog, first_loaded.eog)

timing_summary = pd.DataFrame(
    {
        "load": ["FIF miss", "NPY hit"],
        "seconds": [first_load_seconds, cached_load_seconds],
    }
)
display(timing_summary)
print(f"Observed speed ratio: {first_load_seconds / cached_load_seconds:.1f}x")

,load,seconds
0,FIF miss,0.028401
1,NPY hit,0.020121


Observed speed ratio: 1.4x


### 5. Проверить отдельный namespace для float64

Разные dtype не используют одну cache entry, поэтому преобразования не смешиваются.

In [6]:
float64_dataset = NumpyDataset(
    DATASET_DIR,
    dtype=np.float64,
    cache_policy="disk",
    cache_dir=DEMO_CACHE_ROOT / "float64",
)
float64_loaded = float64_dataset[sample_key]

dtype_summary = pd.DataFrame(
    [
        {
            "dtype": str(item.eeg.dtype),
            "entry": str(owner.get_cache_entry_path(sample_key).relative_to(PROJECT_ROOT)),
            "memory_mib": (item.eeg.nbytes + item.eog.nbytes) / 2**20,
        }
        for owner, item in ((dataset, first_loaded), (float64_dataset, float64_loaded))
    ]
)
display(dtype_summary)

,dtype,entry,memory_mib
0,float32,artifacts/cache/notebook-demo/float32/S_1/Tria...,4.15065
1,float64,artifacts/cache/notebook-demo/float64/S_1/Tria...,8.30130


### 6. Посмотреть автоматический project cache path

Если `cache_dir` не указан, путь определяется по ближайшему `pyproject.toml`. Конструктор не создает файлы до обращения к семплу.

In [7]:
default_cache_dataset = NumpyDataset(DATASET_DIR)

pd.Series(
    {
        "default_cache_dir": str(default_cache_dataset.cache_dir.relative_to(PROJECT_ROOT)),
        "expected": "artifacts/cache/Data_Train/exec/float32",
    },
    name="default_path",
)

default_cache_dir    artifacts/cache/Data_Train/exec/float32
expected             artifacts/cache/Data_Train/exec/float32
Name: default_path, dtype: str

## Checks

Фиксируем корректность записи, manifest и повторного чтения.

In [8]:
assert cache_files == ["eeg.npy", "eog.npy", "manifest.json"]
assert manifest["schema_version"] == NumpyDataset.CACHE_SCHEMA_VERSION
assert manifest["key"] == {"subject_id": 1, "trial_number": 1, "block_index": 1}
assert manifest["dtype"] == "float32"
assert cached_loaded.eeg.dtype == np.float32
assert float64_loaded.eeg.dtype == np.float64
assert dataset.get_cache_entry_path(sample_key) != float64_dataset.get_cache_entry_path(sample_key)
assert default_cache_dataset.cache_dir == PROJECT_ROOT / "artifacts/cache/Data_Train/exec/float32"

pd.Series(
    {
        "atomic_entry_files": "passed",
        "manifest_metadata": "passed",
        "cache_hit_without_mne": "passed",
        "dtype_isolation": "passed",
        "default_path": "passed",
    },
    name="validation",
)

atomic_entry_files       passed
manifest_metadata        passed
cache_hit_without_mne    passed
dtype_isolation          passed
default_path             passed
Name: validation, dtype: str

## Next Steps

Disk cache ускоряет повторные эпохи без роста RAM. Следующий этап добавит ограниченный LRU memory cache и порядок `RAM → disk → FIF` для режима `both`.